# Week 4 - Noise Sensitivity Analysis

## Gaussian Noise Injection

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [ ]:
df = pd.read_csv("../data/cleaned_ai4i.csv")

print(df.shape)
df.head()

In [ ]:
target = "Machine failure"

drop_cols = [
    "UDI",
    "Product ID",
    "Type",
    target
]

X = df.drop(columns=drop_cols, errors="ignore")
y = df[target]

print(X.shape)
print(X.dtypes)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts())
print("\nTest class distribution:")
print(y_test.value_counts())

In [ ]:
def inject_noise(X_test, noise_level):

    X_noisy = X_test.copy()

    numeric_cols = X_noisy.select_dtypes(include=[np.number]).columns

    # Convert numeric columns to float
    X_noisy[numeric_cols] = X_noisy[numeric_cols].astype(float)

    noise = np.random.normal(
        loc=0,
        scale=noise_level,
        size=X_noisy[numeric_cols].shape
    )

    X_noisy.loc[:, numeric_cols] += noise

    return X_noisy

In [ ]:
X_noise = inject_noise(X_test,0.05)

print(X_noise.head())

In [ ]:
print("Original Test Data:")
print(X_test.head())

print("\nNoisy Test Data:")
print(X_noise.head())

In [ ]:
numeric_cols = X_test.select_dtypes(include=[np.number]).columns

difference = X_noise[numeric_cols] - X_test[numeric_cols]

print(difference.head())

## Gaussian Noise Injection

This experiment evaluates the robustness of the trained LightGBM model by adding Gaussian noise to the numerical features of the test dataset.

- Mean (μ) = 0
- Standard Deviation (σ) = noise_level

The noisy dataset simulates sensor measurement errors commonly observed in real-world IoT systems. Comparing the original and noisy datasets helps assess how stable the model remains under different noise conditions before evaluating multiple noise levels in the next steps.

In [ ]:
print("Original Test Shape:", X_test.shape)
print("Noisy Test Shape   :", X_noise.shape)

print("\nMissing values in noisy data:")
print(X_noise.isnull().sum().sum())

print("\nData types:")
print(X_noise.dtypes)

# Week 4 Day 2 - Noise Level Evaluation

This section evaluates the trained model under three Gaussian noise levels:

- Low Noise: 0.05
- Medium Noise: 0.15
- High Noise: 0.30

The objective is to measure Macro F1 degradation after adding noise to the test dataset.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model trained successfully")

In [ ]:
def inject_noise(X_test, noise_level):
    X_noisy = X_test.copy()

    numeric_cols = X_noisy.select_dtypes(include=[np.number]).columns
    X_noisy[numeric_cols] = X_noisy[numeric_cols].astype(float)

    noise = np.random.normal(
        loc=0,
        scale=noise_level,
        size=X_noisy[numeric_cols].shape
    )

    X_noisy.loc[:, numeric_cols] += noise

    return X_noisy

In [ ]:
from sklearn.metrics import f1_score

# Prediction on clean test data
y_pred_clean = model.predict(X_test)

# Macro F1 Score
clean_macro_f1 = f1_score(
    y_test,
    y_pred_clean,
    average="macro"
)

print(f"Clean Test Macro F1: {clean_macro_f1:.4f}")

In [ ]:
noise_levels = {
    "Low": 0.05,
    "Medium": 0.15,
    "High": 0.30
}

print(noise_levels)

In [ ]:
# Apply low Gaussian noise
X_low_noise = inject_noise(X_test, noise_levels["Low"])

# Predict using trained model
y_pred_low = model.predict(X_low_noise)

# Calculate Macro F1
low_macro_f1 = f1_score(
    y_test,
    y_pred_low,
    average="macro"
)

print(f"Low Noise Macro F1: {low_macro_f1:.4f}")

In [ ]:
# Apply medium Gaussian noise
X_medium_noise = inject_noise(X_test, noise_levels["Medium"])

# Predict using trained model
y_pred_medium = model.predict(X_medium_noise)

# Calculate Macro F1
medium_macro_f1 = f1_score(
    y_test,
    y_pred_medium,
    average="macro"
)

print(f"Medium Noise Macro F1: {medium_macro_f1:.4f}")

In [ ]:
# Apply high Gaussian noise
X_high_noise = inject_noise(X_test, noise_levels["High"])

# Predict using trained model
y_pred_high = model.predict(X_high_noise)

# Calculate Macro F1
high_macro_f1 = f1_score(
    y_test,
    y_pred_high,
    average="macro"
)

print(f"High Noise Macro F1: {high_macro_f1:.4f}")

In [ ]:
# Calculate percentage drop in Macro F1 compared to clean test performance

low_drop_pct = ((clean_macro_f1 - low_macro_f1) / clean_macro_f1) * 100

medium_drop_pct = ((clean_macro_f1 - medium_macro_f1) / clean_macro_f1) * 100

high_drop_pct = ((clean_macro_f1 - high_macro_f1) / clean_macro_f1) * 100

print(f"Low Noise Drop    : {low_drop_pct:.2f}%")
print(f"Medium Noise Drop : {medium_drop_pct:.2f}%")
print(f"High Noise Drop   : {high_drop_pct:.2f}%")

In [ ]:
degradation_table = pd.DataFrame({
    "Noise Level": ["Clean", "Low", "Medium", "High"],
    "Noise Std": [0.00, 0.05, 0.15, 0.30],
    "Macro F1": [
        clean_macro_f1,
        low_macro_f1,
        medium_macro_f1,
        high_macro_f1
    ],
    "Drop%": [
        0.00,
        low_drop_pct,
        medium_drop_pct,
        high_drop_pct
    ]
})

degradation_table

## Noise Degradation Analysis

The degradation table compares model performance on the clean test dataset and noisy test datasets with low, medium, and high Gaussian noise levels.

Macro F1 is used to evaluate robustness because the dataset is imbalanced and both failure and non-failure classes need balanced evaluation. The Drop% column shows how much performance decreases compared to the clean baseline.

This analysis helps determine whether the model remains stable when IoT sensor readings contain real-world measurement noise.

# Week 4 Day 3 - Noise Robustness Visualization

This section visualizes the Macro F1 performance across clean, low, medium, and high Gaussian noise conditions.

The chart includes:
- Noise levels on the x-axis
- Macro F1 score on the y-axis
- Target performance line at F1 = 0.85
- Annotation showing performance drop at high noise

In [ ]:
import os
import matplotlib.pyplot as plt

In [ ]:
chart_data = degradation_table.copy()

chart_data["Noise Label"] = [
    "Clean\n(0.00)",
    "Low\n(0.05)",
    "Medium\n(0.15)",
    "High\n(0.30)"
]

chart_data[["Noise Label", "Macro F1", "Drop%"]]

In [ ]:
plt.figure(figsize=(9, 6))

bars = plt.bar(
    chart_data["Noise Label"],
    chart_data["Macro F1"]
)
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.01,
        f"{height:.3f}",
        ha="center",
        fontsize=10
    )

plt.axhline(
    y=0.85,
    linestyle="--",
    label="Target F1 = 0.85"
)

plt.xlabel("Noise Level")
plt.ylabel("Macro F1 Score")
plt.title("Noise Robustness Analysis - Macro F1 vs Noise Level")

plt.ylim(0, 1.05)

plt.annotate(
    f"High Noise Drop\n{high_drop_pct:.2f}%",
    xy=("High\n(0.30)", high_macro_f1),
    xytext=(2.5, high_macro_f1 + 0.12),
    arrowprops=dict(arrowstyle="->"),
    fontsize=10
)
plt.legend()

print("Chart saved as outputs/noise_robustness.png")

plt.show()

## Noise Robustness Chart Analysis

The bar chart visualizes Macro F1 performance across clean, low, medium, and high Gaussian noise conditions.

A horizontal reference line at F1 = 0.85 is added to compare model performance against the target KPI. The annotation on the high-noise bar highlights the maximum observed performance drop.

This visualization helps determine whether the model remains reliable under noisy IoT sensor readings and whether it is suitable for real-world deployment.

# Week 4 Day 4 - Final Noise Robustness Analysis

This section summarizes the overall robustness evaluation performed under different Gaussian noise levels and discusses the deployment readiness of the predictive maintenance model.